# Class-Weighted Multilabel ResNet-18

Three independent outputs are trained with BCE-based loss. Validation chooses the checkpoint; test is used only at the end.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report

sys.path.append(str(Path.cwd()))
from multilabel_utils import (
    CLASS_NAMES,
    LABEL_COLUMNS,
    MultilabelDataset,
    calculate_multilabel_metrics,
    classifier_transform,
    create_resnet18,
    get_device,
    predict_multilabel,
    set_seed,
    train_classifier,
)

SEED = 42
THRESHOLD = 0.5
EPOCHS = 5
set_seed(SEED)
device = get_device()
PROJECT_ROOT = Path.cwd().parents[1]
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "multilabel"
MODEL_DIR = PROJECT_ROOT / "models" / "multilabel"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")
print("Device:", device)
print("Split sizes:", len(train_df), len(val_df), len(test_df))

Device: mps
Split sizes: 1503 302 300


In [2]:
transform = classifier_transform()
generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    MultilabelDataset(train_df, PROJECT_ROOT, transform),
    batch_size=32,
    shuffle=True,
    generator=generator,
)
val_loader = DataLoader(MultilabelDataset(val_df, PROJECT_ROOT, transform), batch_size=32)
test_loader = DataLoader(MultilabelDataset(test_df, PROJECT_ROOT, transform), batch_size=32)

In [3]:
model = create_resnet18(len(LABEL_COLUMNS)).to(device)
positive_counts = train_df[LABEL_COLUMNS].sum()
negative_counts = len(train_df) - positive_counts
pos_weight = torch.tensor(
    (negative_counts / positive_counts).to_numpy(),
    dtype=torch.float32,
    device=device,
)
print("Positive weights:", dict(zip(LABEL_COLUMNS, pos_weight.cpu().numpy())))
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
checkpoint_path = MODEL_DIR / "weighted_best.pth"

history = train_classifier(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    checkpoint_path,
    epochs=EPOCHS,
)

Positive weights: {'Surface_Crack': np.float32(0.1010989), 'Delamination': np.float32(9.29452), 'Pinhole': np.float32(3.198324)}


Epoch 1/5 | Train loss: 0.4057 | Validation loss: 0.3882


Epoch 2/5 | Train loss: 0.1764 | Validation loss: 0.1366


Epoch 3/5 | Train loss: 0.1286 | Validation loss: 1.2557


Epoch 4/5 | Train loss: 0.1568 | Validation loss: 0.1771


Epoch 5/5 | Train loss: 0.0915 | Validation loss: 0.4134


In [4]:
model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
true_labels, probabilities, predictions = predict_multilabel(
    model, test_loader, device, threshold=THRESHOLD
)

print(classification_report(
    true_labels,
    predictions,
    target_names=CLASS_NAMES,
    zero_division=0,
))

metrics = calculate_multilabel_metrics(true_labels, predictions, "Weighted BCE")
metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv(PROCESSED_DIR / "weighted_metrics.csv", index=False)
metrics_df.round(4)

               precision    recall  f1-score   support

Surface Crack       0.98      0.97      0.97       276
 Delamination       0.96      0.85      0.90        26
      Pinhole       0.80      0.95      0.87        73

    micro avg       0.94      0.95      0.95       375
    macro avg       0.91      0.92      0.91       375
 weighted avg       0.94      0.95      0.95       375
  samples avg       0.96      0.97      0.96       375



,Model,Exact Match Accuracy,Hamming Loss,Micro F1,Macro F1,Surface Crack F1,Delamination F1,Pinhole F1
0,Weighted BCE,0.88,0.0456,0.9458,0.9129,0.9727,0.898,0.8679


## Interpretation

Focus on macro F1 and the minority-label F1 scores. Exact-match accuracy requires the entire three-value vector to be correct.